In [ ]:
# %% Deep learning - Section 24.216
#    The RNN class in PyTorch

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.

In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2
import torchvision
import torchvision.transforms as T
import torch.nn.utils         as utils
import random

from torch.utils.data                 import DataLoader,TensorDataset,Dataset,Subset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from matplotlib.gridspec              import GridSpec
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [ ]:
# %% RNN instance

# Parameters
input_size  = 9      # number of features to extract (e.g., number of data channels)
hidden_size = 16     # number of units in the hidden state
num_layers  = 3      # number of vertical stacks of hidden layers (only the final layer gives an output)
act_fun     = 'tanh'
bias        = True

# RNN instance
rnn = nn.RNN(input_size,hidden_size,num_layers,nonlinearity=act_fun,bias=bias)
print(rnn)


In [ ]:
# %% Check sizes

# Data parameters
seq_length = 5
batch_size = 2

# Generate some data
X = torch.rand(seq_length,batch_size,input_size)

# Hidden layer (typically initialized as zeros)
hidden = torch.zeros(num_layers,batch_size,hidden_size)

# Run some data through the model and show the output sizes
y,h = rnn(X,hidden)
print(f' Input shape: {list(X.shape)}')
print(f'Hidden shape: {list(h.shape)}')
print(f'Output shape: {list(y.shape)}')
print()

# Default hidden state is all zeros if nothing specified (h2)
y,h1 = rnn(X,hidden)
print(h1), print()

y,h2 = rnn(X)
print(h2), print()

print(h1-h2)


In [ ]:
# %% Check the learnt parameters and their sizes

for p in rnn.named_parameters():
    if 'weight' in p[0]:
        print(f'{p[0]} has size {list(p[1].shape)}')


In [17]:
# %% RNN model class

class RNN(nn.Module):
    def __init__(self,input_size,num_hidden,num_layers):
        super().__init__()

        # Parameters
        self.input_size  = input_size
        self.num_hidden = num_hidden
        self.num_layers = num_layers

        # RNN layer(s) and output
        self.rnn = nn.RNN(input_size,num_hidden,num_layers)
        self.out = nn.Linear(num_hidden,1)

    def forward(self,x):

        # Initialize hidden state for first input
        hidden = torch.zeros(self.num_layers,batch_size,self.num_hidden)
        print(f'Hidden: {list(hidden.shape)}')

        # Pass through RNN layers
        y,hidden = self.rnn(x,hidden)
        print(f'RNN out: {list(y.shape)}')
        print(f'RNN hid: {list(hidden.shape)}')

        # Pass the RNN output through the fc output layer
        o = self.out(y)
        print(f'Output: {list(o.shape)}')

        return o,hidden


In [ ]:
# %% Check model

# Generate model instance and inspect
net = RNN(input_size,hidden_size,num_layers)
print(net), print()

# Check learnable parameters
for p in net.named_parameters():
    print(f'{p[0]} has size {list(p[1].shape)}')


In [ ]:
# %% Check the model with some data

# Create data
X = torch.rand(seq_length,batch_size,input_size)
y = torch.rand(seq_length,batch_size,1)
yHat,h = net(X)

# Loss
lossfun = nn.MSELoss()
lossfun(yHat,y)


In [ ]:
# %% Exercise 1
#    In the video, I asked about the "l0" from the parameter name "weight_ih_l0". To explore this further,
#    recreate that RNN instance but set the number of layers to 3. Then go through the code again to print
#    out all of the weights matrices. Refer back to the discussion of layers in the previous video. Do you
#    understand the naming system of the weights matrices?

# Stands for the layer index
